In [31]:
from pathlib import Path
from dataclasses import asdict, replace
import json
from dataset import device_for_run, load_store
from reward_model import load_reward
from rl import RLSettings, train_rl
from tracking import start_task


In [32]:
ROOT = Path.cwd() if (Path.cwd() / 'rl.py').exists() else Path.cwd() / 'task2'
DATA = ROOT / 'data' / 'Yambda'
BASE = ROOT / 'checkpoints' / 'gpt2'
LOG_CLEARML = True
EXPERIMENT = 'h4_uncertainty_10_v2'

common = RLSettings(steps=200, batch_size=16, learning_rate=1e-5,
                    eval_every=50, gradient_log_every=50, max_ndcg_drop=0.005,
                    reward_function='ratio', seed=42)
experiments = {
    'h1_reinforce_v2': {'group_size': 1, 'kl_coefficient': 0.0, 'reward_seeds': [42]},
    'h1_kl_v2': {'group_size': 1, 'kl_coefficient': 0.01, 'reward_seeds': [42]},
    'h1_ce_kl_v2': {'group_size': 1, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01, 'reward_seeds': [42]},
    'h2_group4_v2': {'group_size': 4, 'kl_coefficient': 0.01, 'reward_seeds': [42]},
    'h2_group8_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'reward_seeds': [42]},
    'h2_group16_v2': {'group_size': 16, 'kl_coefficient': 0.01, 'reward_seeds': [42]},
    'h2_pairwise_v2': {'group_size': 8, 'pairwise_weight': 0.1, 'kl_coefficient': 0.01, 'reward_seeds': [42]},
    'h3_h4_control_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01,
                         'reward_seeds': [42]},
    'h3_relevance_01_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01,
                           'reward_weight': 1.0, 'relevance_weight': 0.1, 'reward_seeds': [42]},
    'h3_relevance_03_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01,
                           'reward_weight': 1.0, 'relevance_weight': 0.3, 'reward_seeds': [42]},
    'h4_ensemble_control_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01,
                               'uncertainty_penalty': 0.0, 'reward_seeds': [42, 43]},
    'h4_uncertainty_05_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01,
                             'uncertainty_penalty': 0.5, 'reward_seeds': [42, 43]},
    'h4_uncertainty_10_v2': {'group_size': 8, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01,
                             'uncertainty_penalty': 1.0, 'reward_seeds': [42, 43]},
}
if EXPERIMENT not in experiments:
    raise ValueError(f'Unknown experiment: {EXPERIMENT}')
specification = experiments[EXPERIMENT]
settings = replace(common, **{key: value for key, value in specification.items() if key != 'reward_seeds'})
checkpoint = ROOT / 'checkpoints' / EXPERIMENT
if checkpoint.exists():
    raise FileExistsError(f'Choose a new experiment name; checkpoint exists: {checkpoint}')
device = device_for_run()
gpt_settings = json.loads((BASE / 'settings.json').read_text())
store = load_store(DATA, max_length=gpt_settings['max_length'], vocab_path=BASE / 'vocab.json')
print(EXPERIMENT, asdict(settings), 'catalog:', len(store.catalog.items))


h4_uncertainty_10_v2 {'steps': 200, 'batch_size': 16, 'learning_rate': 1e-05, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'temperature': 1.0, 'kl_coefficient': 0.01, 'ce_coefficient': 0.01, 'uncertainty_penalty': 1.0, 'reward_weight': 1.0, 'relevance_weight': 0.0, 'reward_function': 'ratio', 'like_weight': 0.3, 'baseline_decay': 0.9, 'reward_std_floor': 0.05, 'group_size': 8, 'pairwise_weight': 0.0, 'reward_batch_size': 256, 'eval_every': 50, 'gradient_log_every': 50, 'max_ndcg_drop': 0.005, 'seed': 42} catalog: 30000


In [33]:
models = [load_reward(ROOT / 'checkpoints' / f'reward_seed_{seed}', store.catalog.vocab_size, device)[0].cpu()
          for seed in specification['reward_seeds']]
audit, _ = load_reward(ROOT / 'checkpoints' / 'reward_seed_44', store.catalog.vocab_size, device)
audit.cpu()
task = start_task(EXPERIMENT, {'settings': asdict(settings), 'reward_seeds': specification['reward_seeds'],
                               'audit_seed': 44}, LOG_CLEARML)
policy, history = train_rl(store, BASE, models, settings, checkpoint, device, audit_model=audit, task=task)
(checkpoint / 'experiment.json').write_text(json.dumps({'name': EXPERIMENT, 'reward_seeds': specification['reward_seeds'],
                                                       'audit_seed': 44}, indent=2))
if task:
    task.close()
print('best selection:', json.loads((checkpoint / 'selection.json').read_text()))
print('best validation:', json.loads((checkpoint / 'validation_metrics.json').read_text()))


Can't get url information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get branch information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get commit information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages


ClearML Task: created new task id=092e6d91e62141508ff5c18fe9463d11


Can't get diff information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/092e6d91e62141508ff5c18fe9463d11/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

best selection: {'criterion': 'maximum predicted_reward@20 subject to ndcg@20 >= initial - max_ndcg_drop', 'best_step': 200, 'initial_ndcg@20': 0.08449515636346257, 'minimum_ndcg@20': 0.07949515636346256}
best validation: {'queries': 252, 'target_outside_catalog': 73, 'target_previously_seen': 125, 'eligible_queries': 179, 'catalog_items': 30000, 'recall@5': 0.07936507936507936, 'ndcg@5': 0.06398100247337184, 'eligible_recall@5': 0.11173184357541899, 'eligible_ndcg@5': 0.0900738135379313, 'coverage@5': 0.036066666666666664, 'popular_fraction@5': 0.12777777777777777, 'repeat_fraction@5': 0.7658730158730159, 'predicted_reward@5': 0.6332575678825378, 'independent_reward@5': 0.6389669179916382, 'recall@10': 0.10317460317460317, 'ndcg@10': 0.07163351657076185, 'eligible_recall@10': 0.1452513966480447, 'eligible_ndcg@10': 0.10084718533984352, 'coverage@10': 0.06616666666666667, 'popular_fraction@10': 0.13055555555555556, 'repeat_fraction@10': 0.7317460317460317, 'predicted_reward@10': 0.6337

In [34]:
import pandas as pd
history_frame = pd.DataFrame(history)
display(history_frame.tail())


,gradient_norm_policy,gradient_norm_pairwise,gradient_norm_kl,gradient_norm_ce,gradient_norm_total,loss,policy_loss,rl_loss,pairwise_loss,weighted_pairwise_loss,...,val_ndcg@20,val_eligible_recall@20,val_eligible_ndcg@20,val_coverage@20,val_popular_fraction@20,val_repeat_fraction@20,val_predicted_reward@20,val_independent_reward@20,val_kl_vs_base,val_parameter_delta_l2
195,NaN,NaN,NaN,NaN,5.062005,-0.065671,-0.126524,-0.126524,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,NaN,NaN,NaN,NaN,4.735374,0.176924,0.126118,0.126118,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,NaN,NaN,NaN,NaN,4.828447,0.175594,0.118327,0.118327,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,NaN,NaN,NaN,NaN,5.026298,-0.632582,-0.683055,-0.683055,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
199,5.227085,0.0,0.008779,0.018582,5.228344,-0.163288,-0.212906,-0.212906,0.0,0.0,...,0.081606,0.201117,0.114886,0.116,0.130952,0.702778,0.631909,0.63821,0.012132,0.309859


h4_uncertainty_10_v2

In [35]:
cols = [
    'step',
    'val_ndcg@20',
    'val_recall@20',
    'val_predicted_reward@20',
    'val_independent_reward@20',
    'val_kl_vs_base'
]

display(history_frame[cols].dropna().round(6))

,step,val_ndcg@20,val_recall@20,val_predicted_reward@20,val_independent_reward@20,val_kl_vs_base
49,50,0.083538,0.146825,0.630955,0.637981,0.002392
99,100,0.083534,0.146825,0.631347,0.638114,0.004772
149,150,0.082869,0.146825,0.631174,0.638132,0.008275
199,200,0.081606,0.142857,0.631909,0.638210,0.012132
